# Flash Attention: Fast and Memory-Efficient Exact Attention

## Learning Objectives
1. Understand the IO bottleneck in standard attention computation
2. Implement block-wise attention with online softmax
3. Analyze memory usage and speedup from Flash Attention
4. Compare memory profiles: standard vs Flash attention during training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Tuple

# Device setup for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Level 1: Standard Attention (Baseline)

Simplest implementation: compute full NxN attention matrix.

In [ ]:
def standard_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: torch.Tensor = None,
    dropout: float = 0.0
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Standard scaled dot-product attention.
    
    Args:
        query: (B, H, L, D) - queries
        key: (B, H, L, D) - keys
        value: (B, H, L, D) - values
        mask: (B, H, L, L) - attention mask
        dropout: dropout probability
    
    Returns:
        output: (B, H, L, D) - attention output
        attention_weights: (B, H, L, L) - attention weights
    """
    B, H, L, D = query.shape
    
    # Compute attention scores: (B, H, L, L)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (D ** 0.5)
    
    # Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Softmax
    attention = F.softmax(scores, dim=-1)
    
    # Dropout
    if dropout > 0:
        attention = F.dropout(attention, p=dropout, training=True)
    
    # Multiply by values: (B, H, L, D)
    output = torch.matmul(attention, value)
    
    return output, attention

# Test standard attention
batch_size, num_heads, seq_len, head_dim = 2, 4, 64, 64
query = torch.randn(batch_size, num_heads, seq_len, head_dim).to(device)
key = torch.randn(batch_size, num_heads, seq_len, head_dim).to(device)
value = torch.randn(batch_size, num_heads, seq_len, head_dim).to(device)

output_std, attention_std = standard_attention(query, key, value)

print(f"Query shape: {query.shape}")
print(f"Output shape: {output_std.shape}")
print(f"Attention matrix shape: {attention_std.shape}")
print(f"Attention matrix memory: {attention_std.numel() * 4 / 1e6:.2f} MB")
print(f"Total activation memory: {(query.numel() + attention_std.numel() + output_std.numel()) * 4 / 1e6:.2f} MB")


In [ ]:
# Measure memory usage vs sequence length
seq_lengths = [128, 256, 512, 1024, 2048, 4096]
memory_standard = []

for seq_len in seq_lengths:
    query = torch.randn(1, 8, seq_len, 64).to(device)
    key = torch.randn(1, 8, seq_len, 64).to(device)
    value = torch.randn(1, 8, seq_len, 64).to(device)
    
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    output, attn = standard_attention(query, key, value)
    
    # Memory for attention matrix (main memory cost)
    attn_memory = attn.numel() * 4 / 1e9  # in GB
    memory_standard.append(attn_memory)
    
    print(f"Seq_len: {seq_len:4d} | Attention matrix memory: {attn_memory:.4f} GB | "
          f"Theoretical O(N²): {(seq_len**2) * 4 / 1e9:.4f} GB")

# Clear GPU
del query, key, value, output, attn
torch.cuda.empty_cache()


## Level 2: Flash Attention (Block-wise with Online Softmax)

Production implementation: O(N) memory using tiling and online softmax.

In [ ]:
def flash_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    block_size: int = 128,
    causal: bool = False
) -> Tuple[torch.Tensor, dict]:
    """
    Flash Attention: memory-efficient attention with block-wise computation.
    
    Args:
        query: (B, H, L, D) - queries
        key: (B, H, L, D) - keys
        value: (B, H, L, D) - values
        block_size: Size of blocks for tiling
        causal: Whether to use causal masking (for autoregressive)
    
    Returns:
        output: (B, H, L, D) - attention output
        stats: Dictionary with statistics
    """
    B, H, L, D = query.shape
    
    # Validate block size
    block_size = min(block_size, L)
    
    output = torch.zeros_like(query)
    
    # Stats for analysis
    stats = {
        'num_blocks': (L + block_size - 1) // block_size,
        'block_size': block_size,
        'peak_block_memory_mb': 0
    }
    
    # Process in blocks: for each query block
    for block_idx in range(0, L, block_size):
        block_end = min(block_idx + block_size, L)
        block_len = block_end - block_idx
        
        Q_block = query[:, :, block_idx:block_end, :]  # (B, H, block_size, D)
        
        # Initialize online softmax accumulators
        m = torch.full((B, H, block_len), -torch.inf, device=device)  # max per token
        l = torch.zeros((B, H, block_len), device=device)  # sum of exp
        O = torch.zeros((B, H, block_len, D), device=device)  # output accumulator
        
        # Process key blocks (inner loop)
        for key_block_idx in range(0, L, block_size):
            key_block_end = min(key_block_idx + block_size, L)
            key_block_len = key_block_end - key_block_idx
            
            K_block = key[:, :, key_block_idx:key_block_end, :]  # (B, H, block_size, D)
            V_block = value[:, :, key_block_idx:key_block_end, :]  # (B, H, block_size, D)
            
            # Compute attention scores for this block
            S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) / (D ** 0.5)  # (B, H, block_size, block_size)
            
            # Causal mask: prevent attending to future positions
            if causal:
                causal_mask = torch.triu(torch.ones_like(S_block), diagonal=1).bool()
                S_block = S_block.masked_fill(causal_mask, -1e9)
            
            # Track peak memory usage during block computation
            block_memory_mb = (Q_block.numel() + K_block.numel() + V_block.numel() + S_block.numel()) * 4 / 1e6
            stats['peak_block_memory_mb'] = max(stats['peak_block_memory_mb'], block_memory_mb)
            
            # Online softmax: update running statistics
            m_block_old = m.clone()  # Previous max
            m = torch.maximum(m, S_block.max(dim=-1).values)  # New max
            
            # Compute exp with numerical stability
            S_block_scaled = S_block - m.unsqueeze(-1)  # Subtract new max
            P_block = torch.exp(S_block_scaled)  # Attention weights
            
            # Update statistics
            l_new = torch.exp(m_block_old - m) * l + P_block.sum(dim=-1)
            
            # Accumulate output
            O = torch.exp(m_block_old - m).unsqueeze(-1) * O + torch.matmul(P_block, V_block)
            
            l = l_new
        
        # Normalize output: divide by softmax denominator
        O = O / (l.unsqueeze(-1) + 1e-8)
        output[:, :, block_idx:block_end, :] = O
    
    return output, stats

# Test Flash Attention
output_flash, stats = flash_attention(query, key, value, block_size=32)

print(f"Flash Attention output shape: {output_flash.shape}")
print(f"Block configuration:")
print(f"  Number of blocks: {stats['num_blocks']}")
print(f"  Block size: {stats['block_size']}")
print(f"  Peak block memory: {stats['peak_block_memory_mb']:.2f} MB")
print(f"\nMemory comparison:")
print(f"  Standard attention: {attention_std.numel() * 4 / 1e6:.2f} MB (full NxN matrix)")
print(f"  Flash attention: ~{stats['peak_block_memory_mb']:.2f} MB (only current block)")


In [ ]:
# Verify Flash Attention produces similar results to standard attention
with torch.no_grad():
    query_test = torch.randn(2, 4, 256, 64).to(device)
    key_test = torch.randn(2, 4, 256, 64).to(device)
    value_test = torch.randn(2, 4, 256, 64).to(device)
    
    output_std, _ = standard_attention(query_test, key_test, value_test)
    output_flash, _ = flash_attention(query_test, key_test, value_test, block_size=64)
    
    # Check numerical similarity
    max_diff = torch.abs(output_std - output_flash).max().item()
    mean_diff = torch.abs(output_std - output_flash).mean().item()
    
    print(f"Numerical validation:")
    print(f"  Max difference: {max_diff:.6f}")
    print(f"  Mean difference: {mean_diff:.8f}")
    print(f"  Relative error: {mean_diff / (torch.abs(output_std).mean().item() + 1e-8) * 100:.4f}%")
    print(f"\nValidation: {'PASS' if max_diff < 1e-3 else 'FAIL'} (tolerance: 1e-3)")


## Real-World Example 1: Memory Usage Across Sequence Lengths

Measure how Flash Attention saves memory as sequence length increases.

In [ ]:
# Comprehensive memory comparison
seq_lengths = [128, 256, 512, 1024, 2048]
memory_standard_list = []
memory_flash_list = []
time_standard_list = []
time_flash_list = []

print("Benchmarking memory and speed across sequence lengths...")
print("-" * 70)

for seq_len in seq_lengths:
    query = torch.randn(1, 8, seq_len, 64).to(device)
    key = torch.randn(1, 8, seq_len, 64).to(device)
    value = torch.randn(1, 8, seq_len, 64).to(device)
    
    # Standard attention
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    start = time.perf_counter()
    output_std, attn_std = standard_attention(query, key, value)
    torch.cuda.synchronize()
    time_std = time.perf_counter() - start
    mem_std = attn_std.numel() * 4 / 1e9
    
    # Flash attention
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    start = time.perf_counter()
    output_flash, stats = flash_attention(query, key, value, block_size=128)
    torch.cuda.synchronize()
    time_flash = time.perf_counter() - start
    mem_flash = stats['peak_block_memory_mb'] / 1000  # Convert to GB
    
    memory_standard_list.append(mem_std)
    memory_flash_list.append(mem_flash)
    time_standard_list.append(time_std * 1000)  # Convert to ms
    time_flash_list.append(time_flash * 1000)
    
    speedup = time_std / time_flash
    mem_reduction = mem_std / (mem_flash + 1e-8)
    
    print(f"Seq_len {seq_len:4d}: Std {mem_std:.4f} GB | Flash {mem_flash:.4f} GB | "
          f"Reduction {mem_reduction:.1f}x | Speedup {speedup:.2f}x")

# Clear GPU
del query, key, value, output_std, output_flash, attn_std
torch.cuda.empty_cache()

print("-" * 70)


## Real-World Example 2: Transformer Layer with Flash Attention

Benchmark a real transformer layer with standard vs Flash attention.

In [ ]:
class TransformerLayer(nn.Module):
    """Single transformer layer for benchmarking."""
    
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, attention_fn=None):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.attention_fn = attention_fn or standard_attention
        self.head_dim = d_model // num_heads
        
        # Multi-head attention components
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        # Feedforward
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
        
        Returns:
            output: (batch, seq_len, d_model)
        """
        batch_size, seq_len, d_model = x.shape
        
        # Self-attention
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Apply attention (standard or flash)
        if self.attention_fn == flash_attention:
            attn_output, _ = flash_attention(Q, K, V, block_size=128)
        else:
            attn_output, _ = standard_attention(Q, K, V)
        
        # Reshape and project
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, d_model)
        attn_output = self.W_o(attn_output)
        
        # Residual and layer norm
        x = self.norm1(x + attn_output)
        
        # Feedforward
        ff_output = self.ff(x)
        x = self.norm2(x + ff_output)
        
        return x

# Benchmark transformer layers
layer_std = TransformerLayer(d_model=512, attention_fn=standard_attention).to(device)
layer_flash = TransformerLayer(d_model=512, attention_fn=flash_attention).to(device)

# Copy weights so they're identical
layer_flash.load_state_dict(layer_std.state_dict())

batch_size = 2
seq_lengths_model = [256, 512, 1024]

print("\nTransformer Layer Benchmarks:")
print("-" * 70)

for seq_len in seq_lengths_model:
    x = torch.randn(batch_size, seq_len, 512).to(device)
    
    # Standard attention layer
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(5):
            _ = layer_std(x)
    torch.cuda.synchronize()
    time_std = (time.perf_counter() - start) / 5 * 1000
    
    # Flash attention layer
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(5):
            _ = layer_flash(x)
    torch.cuda.synchronize()
    time_flash = (time.perf_counter() - start) / 5 * 1000
    
    speedup = time_std / time_flash
    print(f"Seq_len {seq_len:4d}: Std {time_std:6.2f}ms | Flash {time_flash:6.2f}ms | Speedup {speedup:5.2f}x")

print("-" * 70)


## Real-World Example 3: Training Efficiency with Long Sequences

Demonstrate how Flash Attention enables training with longer sequences.

In [ ]:
# Simulate training a transformer with increasing sequence lengths
# Show memory and speed improvements

def simulate_training_step(layer, x, backward=True):
    """Simulate one training step (forward + backward)."""
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    
    output = layer(x)
    loss = output.mean()
    
    if backward:
        loss.backward()
    
    peak_memory = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    return loss.item(), peak_memory

print("\nTraining with increasing sequence lengths:")
print("-" * 70)
print("Seq_len | Std Memory (GB) | Flash Memory (GB) | Memory Savings |")
print("-" * 70)

seq_lengths_training = [512, 1024, 2048, 4096]

for seq_len in seq_lengths_training:
    x = torch.randn(4, seq_len, 512, requires_grad=True).to(device)
    
    # Standard attention (requires_grad for backward)
    layer_std.train()
    try:
        loss_std, mem_std = simulate_training_step(layer_std, x.detach().clone().requires_grad_(True))
        mem_std_str = f"{mem_std:.2f}"
    except RuntimeError as e:
        mem_std_str = "OOM"
        loss_std = 0
    
    # Flash attention
    layer_flash.train()
    try:
        loss_flash, mem_flash = simulate_training_step(layer_flash, x.detach().clone().requires_grad_(True))
        mem_flash_str = f"{mem_flash:.2f}"
    except RuntimeError as e:
        mem_flash_str = "OOM"
        loss_flash = 0
    
    if mem_std_str != "OOM" and mem_flash_str != "OOM":
        savings = (float(mem_std_str) - float(mem_flash_str)) / float(mem_std_str) * 100 if float(mem_std_str) > 0 else 0
        print(f"{seq_len:6d} | {mem_std_str:14s} | {mem_flash_str:16s} | {savings:13.1f}% |")
    else:
        print(f"{seq_len:6d} | {mem_std_str:14s} | {mem_flash_str:16s} | N/A            |")

print("-" * 70)
print("\nNote: O(N^2) memory in standard attention makes long sequences infeasible.")


## Memory and Speed Comparison Visualization

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Memory usage vs sequence length
seq_lengths_plot = [128, 256, 512, 1024, 2048]
memory_standard_gb = [m for m in memory_standard_list]
memory_flash_gb = [m / 1000 for m in memory_flash_list]

axes[0, 0].plot(seq_lengths_plot, memory_standard_gb, marker='o', linewidth=2, 
                label='Standard Attention', markersize=8)
axes[0, 0].plot(seq_lengths_plot, memory_flash_gb, marker='s', linewidth=2, 
                label='Flash Attention', markersize=8)
axes[0, 0].set_xlabel('Sequence Length', fontsize=11)
axes[0, 0].set_ylabel('Memory (GB)', fontsize=11)
axes[0, 0].set_title('Attention Memory Usage', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_yscale('log')
axes[0, 0].set_xscale('log')

# Plot 2: Speed comparison
axes[0, 1].plot(seq_lengths_plot, time_standard_list, marker='o', linewidth=2, 
                label='Standard Attention', markersize=8)
axes[0, 1].plot(seq_lengths_plot, time_flash_list, marker='s', linewidth=2, 
                label='Flash Attention', markersize=8)
axes[0, 1].set_xlabel('Sequence Length', fontsize=11)
axes[0, 1].set_ylabel('Time (ms)', fontsize=11)
axes[0, 1].set_title('Forward Pass Latency', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Memory reduction ratio
memory_reduction = [m1 / (m2 + 1e-8) for m1, m2 in zip(memory_standard_list, memory_flash_list)]
axes[1, 0].bar(range(len(seq_lengths_plot)), memory_reduction, color='green', alpha=0.7)
axes[1, 0].set_xlabel('Sequence Length', fontsize=11)
axes[1, 0].set_ylabel('Memory Reduction Ratio', fontsize=11)
axes[1, 0].set_title('Memory Savings (Standard / Flash)', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(range(len(seq_lengths_plot)))
axes[1, 0].set_xticklabels(seq_lengths_plot)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Speed improvement
speedup = [t1 / (t2 + 1e-8) for t1, t2 in zip(time_standard_list, time_flash_list)]
axes[1, 1].bar(range(len(seq_lengths_plot)), speedup, color='blue', alpha=0.7)
axes[1, 1].set_xlabel('Sequence Length', fontsize=11)
axes[1, 1].set_ylabel('Speedup Factor', fontsize=11)
axes[1, 1].set_title('Inference Speedup (Standard / Flash)', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(range(len(seq_lengths_plot)))
axes[1, 1].set_xticklabels(seq_lengths_plot)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/flash_attention_analysis.png', dpi=100, bbox_inches='tight')
print("Visualization saved to /tmp/flash_attention_analysis.png")
plt.show()

print("\nKey Findings:")
print(f"  Average memory reduction: {np.mean(memory_reduction):.1f}x")
print(f"  Average speedup: {np.mean(speedup):.2f}x")
print(f"  Memory grows as O(N^2) for standard, O(N) for Flash")


## Key Takeaways

**Core Insight:** Attention is IO-bound, not compute-bound. Flash Attention minimizes memory transfers.

**Memory Comparison:**
| Method | Forward Memory | Backward Memory | Total |
|--------|----------------|-----------------|-------|
| Standard | O(N^2) | O(N^2) | O(N^2) |
| Flash v1 | O(N) | O(N) | O(N) |
| Flash v2 | O(N) | O(N) | O(N) |

**Speedup Analysis:**
- Standard attention: 10-20 percent GPU utilization (memory-bound)
- Flash Attention: 80-90 percent GPU utilization
- Expected speedup: 2-3x for v1, 4-8x for v2

**When to Use:**
- **Flash Attention v2:** Default choice for all modern LLM training (seq_len >= 1024)
- **Standard attention:** Only for short sequences (< 512) or educational purposes
- **Approximate attention:** Sparse/linear attention only for inference or extreme memory constraints

**Practical Implications:**
1. Can train with 2-3x longer sequences on same hardware
2. Enables efficient fine-tuning of large models on consumer GPUs
3. Is exact (not approximate), so no accuracy loss
4. Now standard in HuggingFace transformers (enabled by default)


## Try It Yourself

1. **Extend to Causal Masking:** Modify Flash Attention to support causal masking for autoregressive models. Hint: Mask attending to future positions in the inner loop.

2. **Implement GQA Variant:** Implement Grouped-Query Attention (fewer K,V heads than Q heads) for further efficiency. How much memory does it save?

3. **Benchmark on Different Hardware:** Run the benchmarks on different GPU types (A40, V100, RTX 4090). How does Flash speedup change?

4. **Integrate with Actual Model:** Load a real BERT or GPT model and replace its attention with Flash Attention using transformers library. Measure end-to-end speedup.

5. **Profile Memory Allocations:** Use PyTorch profiler to break down memory usage (weights vs activations vs intermediate). Where does Flash save most memory?
